# Mini-projet : compresser une image

| Module | Durée | Prérequis |
|---|---|---|
| M8 – Décomposition en valeurs singulières et données | 150 min | M6 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le centre de documentation d'un musée numérise des photographies anciennes pour une exposition en ligne, consultée parfois avec une connexion lente. Une développeuse propose de compresser les images avec la SVD, vue dans ce module : on ne garde que les $k$ premières couches. Le responsable veut une décision argumentée.

Votre mission :

1. **choisir $k$** selon un critère de qualité fixé : un PSNR d'au moins 32 dB, une perte peu visible à l'écran (notebook 5.1) ;
2. **comparer** la SVD au format JPEG, à qualité égale ;
3. **empaqueter** la méthode dans une fonction fiable, avec des tests, que la développeuse pourra réutiliser ;
4. rédiger une **recommandation** : faut-il adopter la SVD pour l'exposition ?

L'image de travail est le portrait de Grace Hopper (domaine public, photographie de l'US Navy) déjà rencontré au notebook 5.1, converti en niveaux de gris.

**Activité** : Quelles informations manquent pour décider ? Quels critères, à part la qualité et la taille du fichier, comptent pour une exposition en ligne ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_k_pour_32_dB = 40              # nombre de couches pour atteindre 32 dB (modifiez)
pari_meilleur_format = "SVD"        # "SVD" ou "JPEG", à qualité égale (modifiez)

## ③ Abstraire

Une image en niveaux de gris de $m \times n$ pixels est une matrice. Garder $k$ couches de sa SVD (notebook 8.2) demande de stocker $k$ colonnes de $U$, $k$ valeurs singulières et $k$ lignes de $V^{\mathsf{T}}$.

![Schéma : une image de m fois n pixels est remplacée par U_k, Sigma_k et V_k transposée, soit k fois (m + n + 1) nombres ; le taux de compression est k(m+n+1)/(mn)](../../assets/schemas/schema-stockage.svg)

Pour comparer honnêtement avec un fichier JPEG, on compte des **octets** : chaque nombre de la SVD est stocké sur 2 octets (nombre à virgule en demi-précision), chaque pixel de l'image originale sur 1 octet.

La qualité se mesure par le PSNR (notebook 5.1) :

$$
\text{PSNR} = 10\log_{10}\frac{255^2}{\text{EQM}} .
$$

**Activité** : Pour une image de $600 \times 512$ pixels, à partir de quel $k$ la SVD (2 octets par nombre) prend-elle **plus** de place que l'image non compressée (1 octet par pixel) ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### L'image et ses valeurs singulières

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt

def charger_portrait():
    try:
        from matplotlib import cbook
        with cbook.get_sample_data("grace_hopper.jpg") as f:
            return plt.imread(f).astype(float)
    except Exception:
        y, x = np.mgrid[0:600, 0:512] / 512
        return np.dstack([128 + 100 * np.sin(6 * x) * np.cos(4 * y)] * 3)

couleur = charger_portrait()
gris = couleur.mean(axis=2)
m, n = gris.shape
U, s, Vt = np.linalg.svd(gris, full_matrices=False)

def psnr(image, reference):
    eqm = np.mean((np.clip(image, 0, 255) - reference) ** 2)
    return np.inf if eqm == 0 else 10 * np.log10(255 ** 2 / eqm)

def reconstruire(k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(s, color=LIOUAI["navy"]); ax.set_xlabel("rang i"); ax.set_ylabel("σi (échelle log)"); ax.grid(alpha=0.3)
ax.set_title(f"Valeurs singulières du portrait ({m} × {n})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, k in zip(axes, [5, 20, 50, 100, None]):
    image = gris if k is None else reconstruire(k)
    ax.imshow(np.clip(image, 0, 255), cmap="gray", vmin=0, vmax=255); ax.axis("off")
    ax.set_title("originale" if k is None else f"k = {k} : {psnr(image, gris):.1f} dB")
plt.tight_layout(); plt.show()

### Choisir $k$

In [ ]:
ks = np.arange(1, 301)
qualites = np.array([psnr(reconstruire(k), gris) for k in ks])
k_32 = int(ks[np.argmax(qualites >= 32)])

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(ks, qualites, color=LIOUAI["navy"])
ax.axhline(32, color=LIOUAI["jaune"], ls="--", label="critère : 32 dB")
ax.axvline(k_32, color=LIOUAI["vert"], label=f"k = {k_32}")
ax.set_xlabel("k"); ax.set_ylabel("PSNR (dB)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

octets_svd = lambda k: 2 * k * (m + n + 1)
print(f"k = {k_32} : {octets_svd(k_32) / 1000:.0f} ko, soit {octets_svd(k_32) / (m * n):.0%} de l'image non compressée ({m * n / 1000:.0f} ko)")

### Comparer au JPEG

On enregistre l'image en JPEG avec différents réglages de qualité (bibliothèque Pillow, installée avec Matplotlib), et l'on mesure la taille du fichier et le PSNR.

In [ ]:
from PIL import Image

gris_8 = np.clip(np.round(gris), 0, 255).astype(np.uint8)
jpeg = []
for q in [10, 20, 30, 50, 70, 85, 95]:
    tampon = io.BytesIO()
    Image.fromarray(gris_8).save(tampon, format="JPEG", quality=q)
    relu = np.asarray(Image.open(io.BytesIO(tampon.getvalue()))).astype(float)
    jpeg.append((len(tampon.getvalue()), psnr(relu, gris)))
jpeg = np.array(jpeg)
svd = np.array([(octets_svd(k), psnr(reconstruire(k), gris)) for k in [5, 10, 20, 40, 60, 80, 100, 150, 200]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(svd[:, 0] / 1000, svd[:, 1], "o-", color=LIOUAI["navy"], label="SVD")
ax.plot(jpeg[:, 0] / 1000, jpeg[:, 1], "s-", color=LIOUAI["vert"], label="JPEG")
ax.axhline(32, color=LIOUAI["jaune"], ls="--")
ax.set_xlabel("taille (ko)"); ax.set_ylabel("PSNR (dB)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Qualité selon la taille")
plt.tight_layout(); plt.show()

taille_jpeg_32 = np.interp(32, jpeg[:, 1], jpeg[:, 0])
print(f"Pour 32 dB : SVD {octets_svd(k_32) / 1000:.0f} ko, JPEG environ {taille_jpeg_32 / 1000:.0f} ko")

**Activité** : Le résultat confirme-t-il votre pari ? Le JPEG découpe l'image en petits blocs et utilise d'autres « profils types » que la SVD : des ondulations fixées à l'avance, les mêmes pour toutes les images. Pourquoi est-ce un avantage pour le stockage ?

Indiquez vos réponses et justifiez-les.

### Empaqueter la méthode : une fonction fiable

La développeuse veut une fonction qu'elle pourra appeler sur n'importe quelle image en niveaux de gris, sans surprise. On l'écrit, puis on la **teste** sur des cas dont on connaît la réponse.

In [ ]:
def compresser(image, psnr_min=32.0):
    # Compresse une image en niveaux de gris (tableau 2D, valeurs de 0 à 255) par SVD.
    # Renvoie l'image reconstruite (entiers de 0 à 255), le nombre k de couches et la taille en octets.
    image = np.asarray(image, dtype=float)
    if image.ndim != 2:
        raise ValueError(f"Image en niveaux de gris attendue (2 dimensions), reçu {image.ndim} dimensions.")
    if image.min() < 0 or image.max() > 255:
        raise ValueError("Les valeurs des pixels doivent être comprises entre 0 et 255.")
    U, s, Vt = np.linalg.svd(image, full_matrices=False)
    for k in range(1, len(s) + 1):
        approx = np.clip(np.round(U[:, :k] @ np.diag(s[:k]) @ Vt[:k]), 0, 255)
        if psnr(approx, image) >= psnr_min:
            break
    m, n = image.shape
    return approx.astype(np.uint8), k, 2 * k * (m + n + 1)

In [ ]:
# Tests : chaque ligne vérifie un comportement attendu
image_c, k_c, octets_c = compresser(gris, 32)
assert image_c.shape == gris.shape and image_c.dtype == np.uint8, "forme ou type incorrect"
assert psnr(image_c, gris) >= 32, "critère de qualité non respecté"
assert psnr(np.clip(np.round(reconstruire(k_c - 1)), 0, 255), gris) < 32, "k n'est pas le plus petit possible"

constante = np.full((50, 80), 120.0)
assert compresser(constante)[1] == 1, "une image uniforme doit tenir en une seule couche"

rayures = np.tile(np.where(np.arange(80) % 10 < 5, 30.0, 220.0), (50, 1))
assert compresser(rayures)[1] == 1, "des rayures verticales sont de rang 1"

bruit = np.random.default_rng(0).uniform(0, 255, (60, 60))
assert compresser(bruit, 32)[1] > 40, "le bruit ne devrait presque pas se compresser"

for mauvaise in [couleur, -gris]:
    try:
        compresser(mauvaise)
        raise AssertionError("une image invalide aurait dû être refusée")
    except ValueError:
        pass
print(f"Tous les tests passent. Portrait : k = {k_c}, {octets_c / 1000:.0f} ko.")

### Déboguer : trois erreurs fréquentes

Chacune des cellules suivantes contient une erreur que l'on commet souvent avec les images. Exécutez-les, décrivez le symptôme, trouvez la cause.

In [ ]:
# 1. Calculer avec des entiers sur 8 bits
a = np.array([200, 100], dtype=np.uint8)
b = np.array([100, 200], dtype=np.uint8)
print("a + b =", a + b, "   a - b =", a - b)

In [ ]:
# 2. Oublier de limiter les valeurs avant de convertir
approx = reconstruire(10)
print("valeurs extrêmes de l'approximation :", approx.min().round(1), approx.max().round(1))
converti = approx.astype(np.uint8)
for k in (int(np.argmin(approx)), int(np.argmax(approx))):
    print(f"pixel {k} : {approx.flat[k]:6.1f} avant conversion, {converti.flat[k]:3d} après conversion directe en uint8")

In [ ]:
# 3. Confondre V et V transposée
U_, s_, V_mal_nomme = np.linalg.svd(gris[:100, :100])
essai = U_[:, :5] @ np.diag(s_[:5]) @ V_mal_nomme[:, :5].T          # on prend des colonnes au lieu de lignes
print("PSNR obtenu :", round(psnr(essai, gris[:100, :100]), 1), "dB")

**Activité** : Pour chaque erreur, quel symptôme aurait pu la révéler dans une application réelle ? Laquelle des trois les tests de `compresser` auraient-ils détectée ?

Indiquez vos réponses et justifiez-les.

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : k = {pari_k_pour_32_dB} ; meilleur format : {pari_meilleur_format}")
print(f"Résultats : k = {k_32} ; à 32 dB, SVD {octets_svd(k_32) / 1000:.0f} ko contre JPEG {taille_jpeg_32 / 1000:.0f} ko")

**Activité** :

1. La SVD suffit-elle pour l'exposition, ou faut-il l'abandonner ? Justifiez avec les chiffres.
2. Pourquoi la SVD compresse-t-elle si bien les normales de température du notebook 8.2 et si mal une photographie ? Pensez au rang « utile » de chaque matrice.
3. Dans quels autres usages la décomposition d'une image en couches reste-t-elle intéressante (débruitage, recherche d'images semblables, reconnaissance de visages…) ?

Indiquez vos réponses et justifiez-les.

### Recommandation au musée

Rédigez une recommandation d'une demi-page, sans formule : la question, la méthode d'évaluation, les résultats chiffrés avec un graphique, la décision, et ce qu'il faudrait vérifier encore (autres images, images en couleur, temps de calcul).

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Une seule image.** Toutes les conclusions reposent sur un portrait. Seraient-elles les mêmes pour un dessin au trait, une carte, une photographie de paysage ?
- **Le critère.** Le PSNR mesure une erreur moyenne. Deux images de même PSNR peuvent-elles paraître de qualité très différente à l'œil ?
- **Généraliser : la couleur.** Écrivez une fonction `compresser_couleur(image, k_luminance, k_couleur)` qui convertit l'image en YCbCr (notebook 5.1), compresse $Y$ avec `k_luminance` couches et chacun des canaux $C_b$, $C_r$ avec `k_couleur` couches, puis revient en RVB. Comparez la taille et le PSNR à ceux d'une compression qui utilise `k_luminance` couches pour chacun des canaux R, V, B.

In [ ]:
def compresser_couleur(image, k_luminance, k_couleur):
    # À compléter : renvoie (image reconstruite, nombre d'octets)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
M_ycc = np.array([[0.299, 0.587, 0.114], [-0.168736, -0.331264, 0.5], [0.5, -0.418688, -0.081312]])

def couches(canal, k):
    U, s, Vt = np.linalg.svd(canal, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k]

def compresser_couleur(image, k_luminance, k_couleur):
    m, n, _ = image.shape
    ycc = image @ M_ycc.T
    ycc_c = np.dstack([couches(ycc[:, :, 0], k_luminance), couches(ycc[:, :, 1], k_couleur), couches(ycc[:, :, 2], k_couleur)])
    rvb = np.clip(ycc_c @ np.linalg.inv(M_ycc).T, 0, 255)
    return rvb, 2 * (k_luminance + 2 * k_couleur) * (m + n + 1)

def psnr_couleur(image, reference):
    return 10 * np.log10(255 ** 2 / np.mean((image - reference) ** 2))

k = 80
rvb_k = np.clip(np.dstack([couches(couleur[:, :, c], k) for c in range(3)]), 0, 255)
octets_rvb = 2 * 3 * k * (couleur.shape[0] + couleur.shape[1] + 1)
ycc_k, octets_ycc = compresser_couleur(couleur, k, 10)
print(f"RVB, {k} couches par canal     : {octets_rvb / 1000:.0f} ko, PSNR {psnr_couleur(rvb_k, couleur):.1f} dB")
print(f"YCbCr, {k} pour Y, 10 pour Cb, Cr : {octets_ycc / 1000:.0f} ko, PSNR {psnr_couleur(ycc_k, couleur):.1f} dB")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Implémenter un modèle général sous forme de programme robuste": 1,
    "Décider si les résultats suffisent pour avancer ou abandonner": 1,
    "Résoudre un problème de fonctionnement": 1,
    "Comparer les mérites des outils dans le contexte": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Expliquer par écrit": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M8-projet-compression", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")